# 🇱🇰 Sri Lankan Traffic Sign Recognition

This notebook trains the **Hybrid ConvNeXt + Transformer** model on the generated Sri Lankan traffic sign dataset.

## 1. Setup & Data Preparation

In [ ]:
import os
import sys
from pathlib import Path
import yaml
import torch
import matplotlib.pyplot as plt

# Add src to path
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))

from src.data import get_dataloaders
from src.models.hybrid_model import create_model
from src.training import Trainer, CombinedLoss
from src.training.losses import create_loss_function

print(f"🔹 Project Root: {project_root}")
print(f"🔹 Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

### 1.1 Generate Annotations (Train.csv)
If you haven't run the CSV converter script yet, this cell will do it for you.

In [ ]:
data_dir = project_root / "sri_lankan_traffc_signs/synthetic"
csv_script = project_root / "create_csv_annotations.py"

if not (data_dir / "Train.csv").exists():
    print("🔸 Generating CSV annotations...")
    !python {csv_script} --data-dir {data_dir}
else:
    print("✅ Train.csv already exists.")

## 2. Configuration

In [ ]:
config_path = project_root / "config_sri_lanka.yaml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Override for notebook (optional)
# config['training']['batch_size'] = 16 
# config['training']['epochs'] = 20

print("✅ Configuration Loaded")
print(f"   Classes: {config['dataset']['num_classes']}")
print(f"   Backbone: {config['model']['backbone']}")
print(f"   Epochs: {config['training']['epochs']}")

## 3. Load Data

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=config['dataset']['path'],
    batch_size=config['training']['batch_size'],
    val_split=config['dataset']['val_split'],
    image_size=config['dataset']['image_size'],
    num_workers=0,  # Set to 0 for Windows notebooks to avoid spawn issues
    use_weighted_sampler=True,
    seed=config['seed']
)

print(f"📊 Train batches: {len(train_loader)}")
print(f"📊 Val batches: {len(val_loader)}")

## 4. Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = create_model(config)
model = model.to(device)

# Class weights for loss
class_weights = train_loader.dataset.get_class_weights().to(device)
criterion = create_loss_function(config, class_weights)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    config=config,
    device=device
)

## 5. Training Loop

In [ ]:
print("🚀 Starting Training...")
history = trainer.train()

## 6. Evaluation

In [ ]:
# Plot Loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend()
plt.title('Accuracy')
plt.show()